# First Order Logic in Kanren


In propositional logic, we typically have a very simple mapping between propositions and logic variables: one variable per proposition. In first order logic, this is not so simple because of the variables and quantifiers. As we know, first order logic requires us to provide an interpretation, which means we cannot treat first order logic in the same abstract way that we treated propositional logic. In particular, we will need to:

* Define the predicates.
* Specify what values our variables can take.
* Define the quantifier.

Let's consider a simple problem, express it in first order logic, and then write a Kanren program to solve it.

Anjali, Brian, and Chen graduated from QUB on different days this summer. One of them studied French, one studied Geography, one read History. One scored a good 60%, one acheived a very good 65%, and the other an excellent 70%. 

We have acquired some knowledge about who studied what and what their marks were. However, our knowledge is incomplete. Here is what we know:

1. The person who studied French was awarded 65%.
2. Anjali studied Geography.
3. Brian's mark was 60%.

Let's express this in first order logic. A natural way to approach this would be to define the following predicates:
* $\mathrm{Studied}(\mathrm{Person}, \mathrm{Subject})$
* $\mathrm{Awarded}(\mathrm{Subject}, \mathrm{Mark})$
* $\mathrm{Scored}(\mathrm{Person}, \mathrm{Mark})$

and in these terms our knowledge base would be expressed as:
* $\mathrm{Studied}(\mathrm{Anjali}, \mathrm{Geography})$
* $\mathrm{Awarded}(\mathrm{French}, \mathrm{65\%})$
* $\mathrm{Scored}(\mathrm{Brian}, \mathrm{60\%})$

However, this does not fit very well with the mechanisms available to us. Logic programming (as we have thus far introduced it) requires us to present partial solutions to a problem and then to "complete"
those partial solutions using search to find the complete solutions that are consistent with all of the partial solutions. Following a pairwise representation requires us to infer new relations to complete the solution, and this is not how this approach to logic programming works.

We will instead find it much more convenient to define a predicate $P(n,s,m)$ over variables $n\in\{Anjali, Brian, Chen\}$, $s\in\{French, Geography, History\}$, $m\in\{60\%,65\%,70\%\}$. In these terms, our solutions are *triples* $(n,s,m)$ and our knowledge is constraints on what the triples are allowed to be.

In these terms, our three pieces of knowledge become:


- $(\exists n): [P(n,French,65\%)]$
- $(\exists m): [P(Anjali,Geography,m)]$
- $(\exists s): [P(Brian,s,60\%)]$

Notice that this approach leaves us with free variables that we will solve for. This is consistent with the logic programming approach of treating knowledge as constraints.

We also know that $Chen$, $History$, and $70\%$ need to be somewhere in the solution:


- $(\exists s)(\exists m): [Chen,s,m)]$
- $(\exists n)(\exists m): [P(n,History,m)]$
- $(\exists n)(\exists s): [P(n,s,70\%)]$


Finally, we also know that each possible solution needs to contain three items (one corresponding to each person). This is implicit knowledge that we naturally use; here we will have to explicitly specify it to prevent uncontrolled trivial solutions with free variables.

We now have to write goals that embody our knowledge. How do we go about this?

* The predicate $P$ takes three quantifiers. We will therefore model the solution as a *list of triples*, each member of which contains values of the variable that satisfy the goals. Thus, each member of that list must be a triple.
* Each piece of knowledge must be present (ie `membero`) in the solution.

This is enough for us to build a set of goals:

In [1]:
from kanren import run, var, membero, eq, lall, lany

graduations = var()

goals = lall(
    eq(
        (
            ("Anjali",var(),var()),
            ("Brian",var(),var()),
            ("Chen",var(),var()),
        ),
        graduations
    ),
    membero((var(), "French", "65%"), graduations),
    membero(("Anjali", "Geography", var()), graduations),
    membero(("Brian", var(), "60"), graduations),
    membero((var(), var(), "70"), graduations),
    membero(("Chen", var(), var()), graduations),
    membero((var(), "History", var()), graduations),
)

run(0, graduations, goals)

((('Anjali', 'Geography', '70'),
  ('Brian', 'History', '60'),
  ('Chen', 'French', '65%')),)

Notice that the solution appears six times, once in each possible permutation. We have included nothing in the goals to prevent this and there is no reason to prefer one ordering over another.

Let's now modify this to include another piece of information:

Anjali, Brian, and Chen graduated from QUB on different days this summer. One of them studied French, one studied Geography, one read History. One scored a good 60%, one acheived a very good 65%, and the other an excellent 70%. One of them graduated on the 2nd July, one graduated on the 3rd July, one graduated on the 4th July.

1. The person who studied French scored 65%.
2. Anjali studied Geography.
3. Brian's mark was 60%.
4. Chen graduated on the day before Anjali.
5. Brian graduated on the 4th.

Our existing goals are therefore modified to (for $d\in\{2,3,4\}):

- $(\exists n)(\exists d): [P(n,French,65,d)]$
- $(\exists m)(\exists d): [P(Anjali,Geography,m,d)]$
- $(\exists s)(\exists d): [P(Brian,s,60,d)]$
- $(\exists s)(\exists m)(\exists d): [Chen,s,m,d)]$
- $(\exists n)(\exists m)(\exists d): [P(n,History,m,d)]$
- $(\exists n)(\exists s)(\exists d): [P(n,s,70,d)]$

To these we add some new goals;


- $(\exists n)(\exists s): [P(Anjali,s,n,3)]$
- $(\exists s,s')(\exists m,m')(\exists d): [P(Chen,s,m,d)\land P(Anjali,s',m',d+1)]$
- $(\exists n)(\exists s)(\exists m): [P(n,s,m,2)]$
- $(\exists n)(\exists s)(\exists m): [P(n,s,m,3)]$
- $(\exists n)(\exists s)(\exists m): [P(n,s,m,4)]$

We can now code these up. We will need to make some adjustments to the rules that we already have to accomodate the additional variable. We will also have to figure out how to deal with the new information that something happened "before" something else.

In [2]:
from kanren import conde
goals = lall(
    eq(
        (
            ("Anjali",var(),var(),var()),
            ("Brian",var(),var(),var()),
            ("Chen",var(),var(),var()),
        ),
        graduations
    ),
    membero((var(), "French", "65",var()), graduations),
    membero(("Anjali", "Geography", var(),var()), graduations),
    membero(("Brian", var(), "60",var()), graduations),
    membero((var(), var(), "70",var()), graduations),
    membero(("Chen", var(), var(),var()), graduations),
    membero((var(), "History", var(),var()), graduations),
    membero((var(), var(), var(), 2), graduations),
    membero((var(), var(), var(), 3), graduations),
    membero((var(), var(), var(), 4), graduations),
    membero(("Brian", var(), var(), 4), graduations),
    conde(
        (membero(("Anjali",var(),var(),3),graduations), membero(("Chen",var(),var(),2),graduations)),
        (membero(("Anjali",var(),var(),4),graduations), membero(("Chen",var(),var(),3),graduations))
    )
)

solutions = run(0, graduations, goals)

print(solutions)

((('Anjali', 'Geography', '70', 3), ('Brian', 'History', '60', 4), ('Chen', 'French', '65', 2)),)


Finally, let us consider a variant of this problem where we have a negative constraint:
1. The person who studied French scored 65%.
2. Anjali studied Geography.
3. Brian's mark was 60%.
4. Chen graduated on the day before Anjali.
5. Anjali did not graduate on the 4th.

Kanren does not provide us with a negation operation. There are sound technical reasons for this. Briefly, the "easy" way to implement this is so-called *negation by failure*. This is essentially the idea that if you can't prove a statement to be true then it must be false. This works for propositional logic but is unable to resolve values of variables in first order logic. There are techniques available for *constructive negation*, but these are all super-exponential in time complexity. This means that some creativity is necessary. Here, we adopt the rather crude method of constructing a positive goal by excluding the desired negative goal. This method is not viable for very large problems.

In [3]:
from kanren import conde
goals = lall(
    eq(
        (
            ("Anjali",var(),var(),var()),
            ("Brian",var(),var(),var()),
            ("Chen",var(),var(),var()),
        ),
        graduations
    ),
    membero((var(), "French", "65",var()), graduations),
    membero(("Anjali", "Geography", var(),var()), graduations),
    membero(("Brian", var(), "60",var()), graduations),
    membero((var(), var(), "70",var()), graduations),
    membero(("Chen", var(), var(),var()), graduations),
    membero((var(), "History", var(),var()), graduations),
    membero((var(), var(), var(), 2), graduations),
    membero((var(), var(), var(), 3), graduations),
    membero((var(), var(), var(), 4), graduations),
    conde(
        (membero(("Anjali",var(),var(),3),graduations), membero(("Chen",var(),var(),2),graduations)),
        (membero(("Anjali",var(),var(),4),graduations), membero(("Chen",var(),var(),3),graduations))
    ),
    lany(membero(("Anjali",var(),var(),2),graduations), membero(("Anjali",var(),var(),3),graduations))
)
solutions = run(0, graduations, goals)

# Reconcile repeated solutions (crudely)
solutions = {tuple(sorted(i)) for i in solutions}
print(solutions)


{(('Anjali', 'Geography', '70', 3), ('Brian', 'History', '60', 4), ('Chen', 'French', '65', 2))}
